# Configure your HydroOpsLM agents / 設定 HydroOpsLM agents

## Goal / 目標
Paths are relative to YAML. One controlled reservoir needs one agent; passive outlets have no LLM. Change run_id and output for a new experiment. A model profile is required only for live LLM mode.

路徑以 YAML 所在目錄為基準。每座受控水庫一個 agent；被動出口不配置 LLM。新實驗更改 run_id 與 output；只有真實 LLM 模式需要 model profile。

## Setup / 準備
Python 3.12 with repository dependencies. Local files only; no provider calls. / 使用 Python 3.12 與既有依賴；使用局部檔案，不呼叫 provider。

In [1]:
from pathlib import Path
import json, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/hydrocnhs').is_dir())
sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT))
from llm_reservoir_agent.workflow import load_settings, run_settings, verify_settings_output
from llm_reservoir_agent.verification import verify_run
from unittest.mock import patch
from tutorials.inspect_decision_flow import show_decision_flow
def ensure_offline(config):
    settings = load_settings(ROOT/config)
    assert settings.report['planned_llm_calls'] == 0
    output = Path(settings.report['output'])
    if not output.exists():
        with patch('urllib.request.urlopen', side_effect=AssertionError('network forbidden')):
            run_settings(settings)
    return settings, verify_settings_output(settings)
print('Checkout:', ROOT.name)


Checkout: HydroCNHS-Delaware-Operations


## Steps

### Inspect commented YAML / 閱讀有註解 YAML

In [2]:
print((ROOT/'examples/fe_walter/dkp/experiment.hydroopslm.offline.yaml').read_text())

# HydroOpsLM offline teaching run; original settings/results are preserved.
# FE Walter V3.7 dkp: original prompt builders, synthetic replies, zero model calls.
schema_version: llm-reservoir-experiment.v1
case: ../../../cases/fe_walter_reference/case.json # Frozen hydrology sample; V3.7 physics.
run_id: hydroopslm-fe-walter-dkp        # Change with output before a new run.
output: ../../../runs/hydroopslm-fe-walter-dkp
max_calls: 0                             # This example cannot call a provider.
auditors:
  release_limits:
    type: release_bounds
    version: "1"
agents:
  fe_operator:
    reservoir: FEWalter                  # Reservoir ID in case.json.
    mode: offline                        # Native prompt/audit run on fixture replies.
    strategy: reference_v3_7             # Generic strategy selector; optional vendor plugin.
    reference:
      profile: fe_walter                 # This reference profile is FE-specific.
      method: dkp                   # Original zero-shot

### Validate without a provider / 不連 provider 的設定驗證

In [3]:
for config in ['examples/fe_walter/benchmark/experiment.hydroopslm.offline.yaml','examples/fe_walter/zero_shot/experiment.hydroopslm.offline.yaml','examples/fe_walter/dkp/experiment.hydroopslm.offline.yaml']:
    plan=load_settings(ROOT/config).report
    print(config, {k:plan[k] for k in ['days','agent_count','agent_strategies','planned_llm_calls']})
live=load_settings(ROOT/'examples/fe_walter/dkp/experiment.llm.yaml')
print('Live template temperature:', live.data['models']['reservoir_model']['temperature'])
print('Live readiness:', live.report['live_ready'])

examples/fe_walter/benchmark/experiment.hydroopslm.offline.yaml {'days': 3, 'agent_count': 1, 'agent_strategies': {'fe_operator': 'benchmark'}, 'planned_llm_calls': 0}
examples/fe_walter/zero_shot/experiment.hydroopslm.offline.yaml {'days': 3, 'agent_count': 1, 'agent_strategies': {'fe_operator': 'reference_v3_7'}, 'planned_llm_calls': 0}
examples/fe_walter/dkp/experiment.hydroopslm.offline.yaml {'days': 3, 'agent_count': 1, 'agent_strategies': {'fe_operator': 'reference_v3_7'}, 'planned_llm_calls': 0}
Live template temperature: 0.7
Live readiness: False


### Inspect generic configuration and changing info / 查看通用設定與變動 info

In [4]:
print((ROOT/'tutorials/config/multi_reservoir.hydroopslm.offline.yaml').read_text())
print('Generic template placeholders: $agent_id, $observation, $knowledge, $previous_transition')
print('info is the observation teaching label, not a YAML key.')
print('Generic offline mode bypasses prompt/model; reference offline mode renders prompt and parses fixtures.')
print('Formal settings: docs/USER_GUIDE.md; module interfaces: docs/LLM_MODULES.md')

# HydroOpsLM offline teaching run; original settings/results are preserved.
# Two controlled agents + one passive reservoir. No model requests.
# Prompton's passive outlet/rating is configured in the case; it has no agent here.
# FE Walter/Beltzville meet on Lehigh branches; Prompton is on a separate branch.
# Paths are relative to this YAML. Fixed releases test the interface, not a strategy.
schema_version: llm-reservoir-experiment.v1      # Required configuration format.
case: ../../cases/fe_belt_prompton_offline/case_2016_01.json # Reservoirs, routing, physics and input CSV/hash.
run_id: hydroopslm-three-reservoir-31d-v2      # Experiment ID; change for a new run.
output: ../../runs/hydroopslm-three-reservoir-31d-v2 # New output folder; cannot overwrite.
auditors:                                     # Available versioned decision checks.
  release_limits:                             # Auditor ID selected by agents.
    type: release_bounds                      # Requested release vs

## Checks / 檢查
Review printed inputs, counts, sources and residuals. Synthetic replies validate interfaces only. / 核對輸入、計數、來源與殘差；合成回覆僅驗證介面。

## Next steps / 下一步
check never calls the provider. Actual live execution needs confirmed case readiness, a named installed model, a budget and explicit --allow-network. This tutorial stops at offline verification.

check 不呼叫 provider。真實執行需要 case readiness、已可用模型、額度與 --allow-network；教學只做離線驗證。